In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import statsmodels.api as sm 

In [ ]:
# 1

XLSX = 'kospi_data.xlsx'
SHEETS = ['Financials1', 'Financials2']
FIN_OUT = 'financials.parquet'

HDR = dict(code=7, name=8, item_code=9, item_name=13)
DATA_START = 14
N_ITEM = 27

RENAME = {
    'M111100.M': 'actq', 'M111400.M': 'rectq', 'M111500.M': 'invtq',
    'M111800.M': 'ppentq', 'M111000.M': 'atq', 'M113100.M': 'lctq',
    'M113700.M': 'dlttq', 'M113300.M': 'dlcq', 'M113000.M': 'ltq',
    'M115000.M': 'seqq', 'M121100.M': 'cogsq', 'M121300.M': 'xsgaq',
    'M121000.M': 'saleq', 'M113200.M': 'apq', 'M121500.M': 'opinc',
    'M121900.M': 'xintq', 'M122700.M': 'ibq', 'M114300.M': 'txditcq',
    'M115200.M': 'pstkq', 'M111300.M': 'cheq', 'M131000.M': 'oancfq',
    'M132020.M': 'dpq', 'M442400.M': 'dvt_all', 'M133010.M': 'dvq',
    'M115300.M': 'cstkq', 'M115600.M': 'req', 'M311020.M': 'epsq',
}

def sheet_to_tidy(path, sheet, n_item=N_ITEM):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    codes  = raw.iloc[HDR['code'],      1:].to_numpy()
    cnames = raw.iloc[HDR['name'],      1:].to_numpy()
    icodes = raw.iloc[HDR['item_code'], 1:].to_numpy()
    dates  = raw.iloc[DATA_START:, 0].to_numpy()
    vals   = raw.iloc[DATA_START:, 1:].apply(pd.to_numeric, errors='coerce').to_numpy()

    n_col, n_date = len(codes), len(dates)
    assert n_col % n_item == 0, f'{sheet}: {n_col} not divisible by {n_item}'
    n_comp = n_col // n_item

    icode_blk = icodes.reshape(n_comp, n_item)
    assert (icode_blk == icode_blk[0]).all(), f'{sheet}: item order differs across companies'
    item_code = icode_blk[0]

    comp_code = codes.reshape(n_comp, n_item)[:, 0]
    comp_name = cnames.reshape(n_comp, n_item)[:, 0]

    cube = vals.reshape(n_date, n_comp, n_item)
    flat = cube.reshape(n_date * n_comp, n_item)
    out = pd.DataFrame(flat, columns=list(item_code))
    out.insert(0, 'date', np.repeat(dates, n_comp))
    out.insert(0, 'name', np.tile(comp_name, n_date))
    out.insert(0, 'code', np.tile(comp_code, n_date))
    out['date'] = pd.to_datetime(out['date'], format='%Y-%m') + pd.offsets.MonthEnd(0)
    out = out.rename(columns=RENAME)
    out = out.sort_values(['code', 'date']).reset_index(drop=True)
    return out

def build_financials(path=XLSX, sheets=SHEETS, out=FIN_OUT):
    if Path(out).exists():
        fin = pd.read_parquet(out)
        print(f'{out} exists, loaded. shape {fin.shape}  n_comp {fin["code"].nunique()}')
        return fin
    parts = []
    for sh in sheets:
        cache = f'{sh}.parquet'
        if Path(cache).exists():
            tidy = pd.read_parquet(cache)
            print(f'  {cache} exists, loaded  ({tidy["code"].nunique()} comps)')
        else:
            tidy = sheet_to_tidy(path, sh)
            tidy.to_parquet(cache)
            print(f'  {sh} -> {cache}  shape {tidy.shape}  n_comp {tidy["code"].nunique()}')
        parts.append(tidy)
    fin = pd.concat(parts, ignore_index=True)
    before = len(fin)
    fin = fin.drop_duplicates(subset=['code', 'date']).reset_index(drop=True)
    fin = fin.sort_values(['code', 'date']).reset_index(drop=True)
    fin.to_parquet(out)
    print(f'\n{out} written. shape {fin.shape}  n_comp {fin["code"].nunique()}  '
          f'n_date {fin["date"].nunique()}  (dropped {before-len(fin)} dup rows)')
    return fin

financials = build_financials()
print('\nexpected: n_comp 1118, n_date 147')
print('columns:', financials.columns.tolist())

  Financials1 -> Financials1.parquet  shape (89082, 30)  n_comp 606
  Financials2 -> Financials2.parquet  shape (75264, 30)  n_comp 512

financials.parquet written. shape (164346, 30)  n_comp 1118  n_date 147  (dropped 0 dup rows)

expected: n_comp 1118, n_date 147
columns: ['code', 'name', 'date', 'actq', 'rectq', 'invtq', 'ppentq', 'atq', 'lctq', 'dlttq', 'dlcq', 'ltq', 'seqq', 'cogsq', 'xsgaq', 'saleq', 'apq', 'opinc', 'xintq', 'ibq', 'txditcq', 'pstkq', 'cheq', 'oancfq', 'dpq', 'dvt_all', 'dvq', 'cstkq', 'req', 'epsq']


In [ ]:
# 2

FVOL_COLS = ['actq', 'rectq', 'invtq', 'ppentq', 'atq', 'lctq', 'dlcq',
             'apq', 'dlttq', 'ltq', 'seqq', 'xoprq', 'cogsq', 'xsgaq']

def compute_fvol(df, fvol_cols=FVOL_COLS, diff_lag=4, win=8, min_obs=4):
    d = df.sort_values(['code', 'date']).reset_index(drop=True).copy()
    if 'xoprq' not in d.columns:
        d['xoprq'] = d['cogsq'] + d['xsgaq']

    pq = pd.PeriodIndex(d['date'], freq='Q')
    d['qidx'] = pq.year * 4 + (pq.quarter - 1)

    g = d.groupby('code', sort=False)
    has_lag = (d['qidx'] - g['qidx'].shift(diff_lag)) == diff_lag
    for c in fvol_cols:
        d[f'd_{c}'] = np.where(has_lag, d[c] - g[c].shift(diff_lag), np.nan)

    g = d.groupby('code', sort=False)
    for c in fvol_cols:
        d[f'std_{c}'] = (g[f'd_{c}'].apply(lambda s: s.rolling(win, min_periods=min_obs).std())
                         .reset_index(level=0, drop=True))

    d['saleq_pos'] = d['saleq'].where(d['saleq'] > 0)
    g = d.groupby('code', sort=False)
    d['avg_sales'] = (g['saleq_pos'].apply(lambda s: s.rolling(win, min_periods=min_obs).mean())
                      .reset_index(level=0, drop=True))
    d['avg_sales_valid'] = d['avg_sales'].where(d['avg_sales'] > 0)

    ind = []
    for c in fvol_cols:
        d[f'fvol_{c}'] = d[f'std_{c}'] / d['avg_sales_valid']
        ind.append(f'fvol_{c}')
    return d, ind



def finalize_fvol(d, ind, fvol_cols=FVOL_COLS, min_valid=10):
    d = d.copy()
    rcols = [f'rank_{c}' for c in fvol_cols]
    ranked = d.groupby('date')[ind].rank(pct=True)
    ranked.columns = rcols
    d = pd.concat([d, ranked], axis=1)
    d['n_valid_rank'] = d[rcols].notna().sum(axis=1)
    d['FVOL'] = d[rcols].mean(axis=1)
    d.loc[d['n_valid_rank'] < min_valid, 'FVOL'] = np.nan
    return d

comp_fvol, ind = compute_fvol(financials)
comp_fvol = finalize_fvol(comp_fvol, ind)
fvol = comp_fvol[comp_fvol['FVOL'].notna()][['code', 'date', 'FVOL']].copy()

print('comp_fvol:', comp_fvol.shape)
print('valid FVOL firm-quarters:', len(fvol))
print('avg firms per quarter:', round(fvol.groupby('date').size().mean(), 1))
print('range:', fvol['date'].min().date(), '~', fvol['date'].max().date())
print('FVOL describe:', fvol['FVOL'].describe().round(3).to_dict())
print('\nfirms/quarter at 5 points:')
gq = fvol.groupby('date').size()
print(gq.iloc[[0, len(gq)//4, len(gq)//2, 3*len(gq)//4, -1]].to_string())

comp_fvol: (164346, 93)
valid FVOL firm-quarters: 68394
avg firms per quarter: 683.9
range: 2001-09-30 ~ 2026-06-30
FVOL describe: {'count': 68394.0, 'mean': 0.501, 'std': 0.203, 'min': 0.021, '25%': 0.345, '50%': 0.482, '75%': 0.641, 'max': 1.0}

firms/quarter at 5 points:
date
2001-09-30     13
2007-12-31    646
2014-03-31    681
2020-06-30    739
2026-06-30    770
